# Part I - (Ford GoBike Trip Data Exploration)
## by Saumyadeep

## Introduction
>This project explores the Ford GoBike System trip data from February 2019 in the San Francisco Bay Area. The dataset contains over 183,000 individual bike trips with information on duration, timing, stations, and rider demographics. The goal is to uncover patterns in ride behavior — specifically what factors influence trip duration — through systematic univariate, bivariate, and multivariate exploration.

## Preliminary Wrangling

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('201902-fordgobike-tripdata (1).csv')

In [ ]:
# Missing values summary
print("MISSING VALUES:")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print(f"\n{'=' * 60}")
print("NUMERIC COLUMNS SUMMARY:")
print("=" * 60)
df.describe()

In [ ]:
print("MAIN FEATURE OF INTEREST: duration_sec (Trip Duration)")
print("=" * 60)
print(f"Mean:   {df['duration_sec'].mean():.1f} seconds ({df['duration_sec'].mean()/60:.1f} minutes)")
print(f"Median: {df['duration_sec'].median():.1f} seconds ({df['duration_sec'].median()/60:.1f} minutes)")
print(f"Min:    {df['duration_sec'].min()} seconds")
print(f"Max:    {df['duration_sec'].max()} seconds ({df['duration_sec'].max()/3600:.1f} hours)")
print(f"Std:    {df['duration_sec'].std():.1f} seconds")

# Check for extreme outliers
q99 = df['duration_sec'].quantile(0.99)
print(f"\n99th percentile: {q99:.0f} seconds ({q99/60:.0f} minutes)")
print(f"Trips > 1 hour: {(df['duration_sec'] > 3600).sum()} ({(df['duration_sec'] > 3600).mean()*100:.2f}%)")

In [ ]:
print("SUPPORTING FEATURES:")
print("=" * 60)

# User type distribution
print("\n1. user_type (Customer vs Subscriber):")
print(df['user_type'].value_counts())
print(f"   Ratio: {df['user_type'].value_counts(normalize=True).round(3).to_dict()}")

# Gender distribution
print("\n2. member_gender:")
print(df['member_gender'].value_counts())

# Bike share for all
print("\n3. bike_share_for_all_trip:")
print(df['bike_share_for_all_trip'].value_counts())

# Derive age from birth year
df['member_age'] = 2019 - df['member_birth_year']
print(f"\n4. member_age (derived from birth year):")
print(f"   Mean age: {df['member_age'].mean():.1f}")
print(f"   Median age: {df['member_age'].median():.1f}")
print(f"   Range: {df['member_age'].min():.0f} - {df['member_age'].max():.0f}")

# Extract time features
df['start_time'] = pd.to_datetime(df['start_time'])
df['hour'] = df['start_time'].dt.hour
df['day_of_week'] = df['start_time'].dt.day_name()
df['duration_min'] = df['duration_sec'] / 60

print(f"\n5. hour (extracted from start_time):")
print(f"   Peak hours: {df['hour'].value_counts().head(3).index.tolist()}")

print(f"\n6. day_of_week (extracted from start_time):")
print(f"   Busiest days: {df['day_of_week'].value_counts().head(3).index.tolist()}")

### What is the structure of your dataset?

> The dataset has 183,412 rows and 16 columns covering Ford GoBike trips in February 2019. Columns include trip duration (seconds), start/end timestamps, station info (ID, name, lat/long), bike ID, user type, member demographics (birth year, gender), and bike-share-for-all status. Missing values exist in station fields (0.11%) and member demographics (4.51% — likely casual "Customer" users without accounts).

### What is/are the main feature(s) of interest in your dataset?

> duration_sec — trip duration. Mean is 12.1 minutes, median 8.6 minutes (right-skewed). 99% of trips are under 58 minutes, with a few outliers up to 23.7 hours. Secondary interest: user_type, since 89.2% are Subscribers (commuters) vs 10.8% Customers (casual users).

### What features in the dataset do you think will help support your investigation into your feature(s) of interest?

> user_type — 89% Subscriber vs 11% Customer (expect different duration patterns)
hour — Peak hours at 8, 17, 18 (confirms commuter-heavy usage)
day_of_week — Busiest on Thu/Tue/Wed (weekday-dominant = commuters)
member_gender — 75% Male, 23% Female, 2% Other
member_age — Median 32 years (some outliers up to 141 need cleaning)
bike_share_for_all_trip — 9.5% are equity program trips

## 1. Univariate Exploration

### GOAL: Finding the distribution of trip durations. (Histogram)

In [ ]:
# Filter to trips under 60 min (99th percentile) to avoid extreme outlier distortion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Raw distribution (log scale)
axes[0].hist(df['duration_sec'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Trip Duration — Full Range (Right-Skewed)')
axes[0].axvline(df['duration_sec'].median(), color='red', linestyle='--', label=f"Median: {df['duration_sec'].median()/60:.1f} min")
axes[0].legend()

# Right: Zoomed in (under 60 min = 99% of data)
duration_clean = df[df['duration_sec'] <= 3600]['duration_sec'] / 60  # convert to minutes
axes[1].hist(duration_clean, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_ylabel('Count')
axes[1].set_title('Trip Duration — Under 60 min (99% of trips)')
axes[1].axvline(duration_clean.median(), color='red', linestyle='--', label=f"Median: {duration_clean.median():.1f} min")
axes[1].legend()

plt.tight_layout()
plt.show()

#### OBSERVATION: Distribution is heavily right-skewed. Most trips are 5-15 minutes
#### (median ~8.6 min), indicating short commuter rides. Outliers beyond 60 min 
#### are rare (<1%) and likely represent forgotten undocked bikes or leisure rides.

### GOAL: Possible breakdown of user types (Subscriber vs Customer)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart - User Type
user_counts = df['user_type'].value_counts()
bars = axes[0].bar(user_counts.index, user_counts.values, color=['#2196F3', '#FF9800'], edgecolor='black')
axes[0].set_xlabel('User Type')
axes[0].set_ylabel('Number of Trips')
axes[0].set_title('Trip Count by User Type')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{bar.get_height():,}\n({bar.get_height()/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)

# Bar chart - Gender
gender_counts = df['member_gender'].value_counts()
bars2 = axes[1].bar(gender_counts.index, gender_counts.values, 
                    color=['#4CAF50', '#E91E63', '#9C27B0'], edgecolor='black')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Number of Trips')
axes[1].set_title('Trip Count by Gender')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{bar.get_height():,}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

#### OBSERVATION: Subscribers dominate at 89.2% — this is primarily a commuter service.
#### Male riders account for ~75% of trips. Notable gender imbalance.

### GOAL: Finnding the exact time durations when most trips occur. (Hour & Day patterns)

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hour of day
axes[0].hist(df['hour'], bins=24, range=(0, 24), edgecolor='black', alpha=0.7, color='teal')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Trips')
axes[0].set_title('Trip Distribution by Hour of Day')
axes[0].set_xticks(range(0, 24, 2))
axes[0].axvline(8, color='red', linestyle='--', alpha=0.5, label='8 AM')
axes[0].axvline(17, color='red', linestyle='--', alpha=0.5, label='5 PM')
axes[0].legend()

# Day of week (ordered)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['day_of_week'].value_counts().reindex(day_order)
axes[1].bar(range(7), day_counts.values, color='teal', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Number of Trips')
axes[1].set_title('Trip Distribution by Day of Week')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.tight_layout()
plt.show()

#### OBSERVATION: Clear commuter pattern — two peaks at 8AM and 5PM (rush hours).
#### Weekdays have significantly more trips than weekends, confirming commuter-dominant usage.
#### Thursday is the busiest whereas Saturday and Sunday are the quietest.

### GOAL: Finding riders age distribution.

In [ ]:
# Clean ages (remove impossible values > 80)
age_clean = df[df['member_age'] <= 80]['member_age']

plt.figure(figsize=(10, 5))
plt.hist(age_clean, bins=30, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('Age (years)')
plt.ylabel('Number of Trips')
plt.title('Age Distribution of Riders (ages ≤ 80)')
plt.axvline(age_clean.median(), color='black', linestyle='--', 
            label=f'Median: {age_clean.median():.0f} years')
plt.legend()
plt.show()

print(f"Ages > 80 removed as outliers: {(df['member_age'] > 80).sum()} trips")

#### OBSERVATION: Riders are predominantly young adults — median age 32.
#### Distribution peaks around 25-35 (millennials). Very few riders above 60.
#### 84 records show age > 80 (likely fake birth years) — these should be cleaned.

### Discuss the distribution(s) of your variable(s) of interest. Were there any unusual points? Did you need to perform any transformations?

> The main variable, duration_sec, is heavily right-skewed with a median of 8.6 minutes but a max of 23.7 hours. 99% of trips fall under 58 minutes, while the remaining 1% (1,710 trips) are extreme outliers — likely forgotten undocked bikes. I applied a log-scale view and filtered to trips ≤ 60 minutes to reveal the true distribution shape, which peaks around 5–15 minutes. This confirms the service is used primarily for short commuter trips, not long leisure rides.

### Of the features you investigated, were there any unusual distributions? Did you perform any operations on the data to tidy, adjust, or change the form of the data? If so, why did you do this?

> Yes, several unusual patterns required cleaning:

> Trip duration — Right-skewed with extreme outliers (up to 23.7 hrs). Filtered to ≤ 2 hours for analysis to prevent distortion.

> Member age — 84 records show ages above 80 (max 141 years), clearly fake birth years. Removed ages > 80 as invalid data.

> Missing values — member_birth_year and member_gender are missing for 4.51% of rows (8,265 records), almost entirely from "Customer" (non-member) users who don't provide demographics.

> Time features — Extracted hour, day_of_week, and duration_min from raw timestamps and seconds to make patterns interpretable.

> User type imbalance — 89.2% Subscribers vs 10.8% Customers. Not cleaned, but noted for context when comparing groups (unequal sample sizes).

## 2. Bivariate Exploration

### GOAL: Finding if there is relationship between trip duration and rider age.

In [ ]:
# Use cleaned data (duration ≤ 60 min, age ≤ 80) — NaN ages removed since age is plotted
df_bi = df[(df['duration_sec'] <= 3600) & (df['member_age'] <= 80)].dropna(subset=['member_age']).copy()
df_bi['member_age'] = df_bi['member_age'].astype(int)

plt.figure(figsize=(14, 6))
sns.stripplot(data=df_bi, x='member_age', y='duration_min',
              size=1, jitter=0.35, native_scale=True)
plt.xlabel('Rider Age (years)')
plt.ylabel('Trip Duration (minutes)')
plt.title(f'Trip Duration vs Rider Age (n={len(df_bi):,}, jittered)')
plt.axhline(df_bi['duration_min'].median(), color='red', linestyle='--',
            alpha=0.7, label=f"Median duration: {df_bi['duration_min'].median():.1f} min")
plt.xticks(fontsize=8)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.stripplot(data=df_bi, x='member_age', y='duration_min', hue='user_type',
              size=1, jitter=0.35, dodge=True, rasterized=True)
plt.xlabel('Age')
plt.ylabel('Trip Duration (Minutes)')
plt.title('Relationship Between Age & Trip Duration, By User Type\n'
          f'(trips over 60 min and ages over 80 excluded — n={len(df_bi):,})')
plt.legend(markerscale=4)
plt.xticks(fontsize=8)
plt.show()

#### OBSERVATION: No strong linear relationship between age and duration.
#### Most trips cluster between 5-15 min regardless of age.

### GOAL: Subscribers and Customers trip duration gap(even if variation w.r.t gender)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot: Duration by User Type
df_bi.boxplot(column='duration_min', by='user_type', ax=axes[0],
              patch_artist=True, 
              boxprops=dict(facecolor='lightblue'),
              medianprops=dict(color='red', linewidth=2))
axes[0].set_xlabel('User Type')
axes[0].set_ylabel('Trip Duration (minutes)')
axes[0].set_title('Trip Duration by User Type')
axes[0].set_ylim(0, 40)  # Focus on main distribution
plt.sca(axes[0])
plt.xticks([1, 2], ['Customer', 'Subscriber'])

# Box plot: Duration by Gender
gender_data = df_bi[df_bi['member_gender'].isin(['Male', 'Female', 'Other'])]
gender_data.boxplot(column='duration_min', by='member_gender', ax=axes[1],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightyellow'),
                    medianprops=dict(color='red', linewidth=2))
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Trip Duration (minutes)')
axes[1].set_title('Trip Duration by Gender')
axes[1].set_ylim(0, 40)

plt.suptitle('')  # Remove auto-title from boxplot
plt.tight_layout()
plt.show()

# Print medians for clarity
print("Median duration (minutes):")
print(df_bi.groupby('user_type')['duration_min'].median())
print()
print(gender_data.groupby('member_gender')['duration_min'].median())

#### OBSERVATION: Customers ride ~2x longer (median ~12 min) than Subscribers (median ~8 min).
#### This confirms Subscribers are commuters (short, routine trips) while Customers are
#### casual/tourist riders (longer, exploratory trips). Gender shows minimal difference.

### GOAL: Usage pattern difference (weekday vs weekend) w.r.t user type

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Create grouped counts
grouped = df.groupby(['day_of_week', 'user_type']).size().unstack(fill_value=0)
grouped = grouped.reindex(day_order)

# Clustered bar chart
ax = grouped.plot(kind='bar', figsize=(12, 6), color=['#FF9800', '#2196F3'], 
                  edgecolor='black', width=0.7)
plt.xlabel('Day of Week')
plt.ylabel('Number of Trips')
plt.title('Trip Count by Day of Week and User Type')
plt.xticks(rotation=45, ha='right')
plt.legend(title='User Type')
plt.tight_layout()
plt.show()


#### OBSERVATION: Subscribers dominate weekdays with a sharp drop on weekends.
#### Customers show relatively flat usage across the week with a slight weekend increase.
#### This confirms: Subscribers = commuters (Mon-Fri), Customers = leisure riders.

### GOAL: Finding trip counts difference across hour of day AND day of week

In [ ]:
# Create pivot table
heatmap_data = df.pivot_table(index='day_of_week', columns='hour', 
                              values='duration_sec', aggfunc='count')
heatmap_data = heatmap_data.reindex(['Monday', 'Tuesday', 'Wednesday', 
                                      'Thursday', 'Friday', 'Saturday', 'Sunday'])

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, fmt='.0f',
            linewidths=0.5, cbar_kws={'label': 'Number of Trips'})
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.title('Trip Frequency Heatmap: Hour vs Day of Week')
plt.tight_layout()
plt.show()

#### OBSERVATION: Hotspots at 8AM and 5PM on weekdays (commuter rush hours).
#### Weekends show a single broad midday peak (11AM-3PM) instead of dual peaks.
#### Thursday 5PM is the single busiest slot. Late nights (11PM-5AM) are nearly empty.

### Talk about some of the relationships you observed in this part of the investigation. How did the feature(s) of interest vary with other features in the dataset?

> Trip duration (the main feature) varies most strongly with user type: Customers ride ~2x longer than Subscribers (median 12 vs 8 min), confirming that Subscribers take short commuter trips while Customers take longer exploratory rides. Duration showed no meaningful relationship with age — most trips cluster at 5–15 minutes regardless of rider age, though younger riders (20–35) have slightly more variability. Gender also had minimal effect on duration. The strongest temporal pattern is that Subscriber trips concentrate around rush hours (8AM, 5PM) on weekdays, while Customer trips spread evenly throughout the day and slightly increase on weekends.

### Did you observe any interesting relationships between the other features (not the main feature(s) of interest)?

> Yes — the user type × day of week relationship was striking. Subscribers drop sharply on weekends (from ~25K+ trips/weekday down to ~8K on Sunday), while Customers remain relatively flat or even increase slightly on weekends. This two-population behavior also appears in the hour × day heatmap: weekdays show a clear dual-peak pattern (8AM and 5PM commuter rush), but weekends show a completely different single broad midday peak (11AM–3PM). This suggests the bike-share system serves two fundamentally different populations — weekday commuters and weekend leisure users — with almost no overlap in their usage patterns. Additionally, the Bike Share for All program users (9.5%) appear almost exclusively in the Subscriber category, suggesting the equity program primarily benefits regular commuters rather than casual riders.

## 3. Multivariate Exploration

### GOAL: Trip duration distribution difference between user types on weekdays vs weekends

In [ ]:
# Create weekday/weekend category
df_multi = df[(df['duration_sec'] <= 3600) & (df['member_age'] <= 80)].copy()
df_multi['day_type'] = df_multi['day_of_week'].apply(
    lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday')

# Facet plot: 2x2 grid (User Type × Day Type)
g = sns.FacetGrid(df_multi, col='user_type', row='day_type', 
                  height=4, aspect=1.5, margin_titles=True, sharey = False)
g.map(plt.hist, 'duration_min', bins=30, color='steelblue', edgecolor='black', alpha=0.7)
g.set_axis_labels('Trip Duration (minutes)', 'Count')
g.set_titles(row_template='{row_name}', col_template='{col_name}')
g.fig.suptitle('Trip Duration Distribution: User Type × Day Type', y=1.02, fontsize=14)

# Add median lines
for ax, (row_val, col_val) in zip(g.axes.flat, 
    [(dt, ut) for dt in ['Weekday', 'Weekend'] for ut in ['Customer', 'Subscriber']]):
    cell = df_multi[(df_multi['day_type'] == row_val) & (df_multi['user_type'] == col_val)]
    med = cell['duration_min'].median()
    ax.axvline(med, color='red', linestyle='--', label=f'Median: {med:.1f} min')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

#### OBSERVATION: Subscribers maintain short, consistent rides (~8-9 min median) 
#### regardless of weekday/weekend. Customers ride longer on weekends (median ~14 min)
#### than weekdays (~11 min), confirming leisure/tourist behavior amplified on weekends.

### GOAL: Age, duration, user type, and gender interaction together

In [ ]:

df_en = df_multi[df_multi['member_gender'].isin(['Male', 'Female'])].copy()

plt.figure(figsize=(12, 7))

# Color = user_type, Marker = gender
colors = {'Subscriber': '#2196F3', 'Customer': '#FF9800'}
markers = {'Male': 'o', 'Female': '^'}

for user in ['Subscriber', 'Customer']:
    for gender in ['Male', 'Female']:
        grp = df_en[(df_en['user_type'] == user) & (df_en['member_gender'] == gender)]
        plt.scatter(grp['member_age'], grp['duration_min'],
            c=colors[user], marker=markers[gender],
            alpha=0.12, s=8, label=f'{user} - {gender}')

plt.xlabel('Rider Age (years)', fontsize=12)
plt.ylabel('Trip Duration (minutes)', fontsize=12)
plt.title('Trip Duration vs Age by User Type & Gender', fontsize=14)
plt.legend(title='User Type - Gender', markerscale=2, framealpha=0.9)
plt.ylim(0, 45)
plt.tight_layout()
plt.show()

### EVEN AFTER MULTIPLE CHANGES THE SCATTER PLOT IS NOT PROPER, SO HEXABIN WITH FACETS WILL WORK BETTER HERE

In [ ]:
df_en = df_multi[df_multi['member_gender'].isin(['Male', 'Female'])].copy()

g = sns.FacetGrid(df_en, col='member_gender', row='user_type',
                  height=4, aspect=1.4, margin_titles=True)
g.map(plt.hexbin, 'member_age', 'duration_min',
      gridsize=30, cmap='YlOrRd', mincnt=1)
g.set_axis_labels('Rider Age (years)', 'Trip Duration (minutes)')
g.set_titles(row_template='{row_name}', col_template='{col_name}')
g.figure.suptitle('Trip Density: Age vs Duration by User Type & Gender',
                  y=1.03, fontsize=14)
plt.tight_layout()
plt.show()

#### OBSERVATION: The Customer–Subscriber duration gap holds across nearly all ages and
#### both genders. Orange (Customer) points sit consistently higher than blue (Subscriber),
#### with only two exceptions at Male ages 59–60 where Customer sample sizes are minimal.
#### Gender (circle vs triangle) does not create meaningful separation — the marker shapes
#### overlap almost entirely within each colour. Age shows no linear trend; the cloud runs
#### flat across the full range. User type dominates; demographics add almost nothing.

### GOAL: Finding hourly usage patterns difference across user type AND weekday/weekend

In [ ]:
# Aggregate by hour, user_type, day_type
hourly = df_multi.groupby(['hour', 'user_type', 'day_type']).size().reset_index(name='trips')

g = sns.FacetGrid(hourly, col='day_type', hue='user_type', 
                  height=5, aspect=1.3, palette=['#FF9800', '#2196F3'])
g.map(plt.plot, 'hour', 'trips', linewidth=2)
g.map(plt.fill_between, 'hour', 'trips', alpha=0.1)
g.set_axis_labels('Hour of Day', 'Number of Trips')
g.set_titles('{col_name}')
g.add_legend(title='User Type')
g.fig.suptitle('Hourly Trip Patterns: Weekday vs Weekend by User Type', y=1.02, fontsize=14)

for ax in g.axes.flat:
    ax.set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

In [ ]:
df_multi['hour_band'] = pd.cut(df_multi['hour'], bins=[0, 6, 10, 16, 20, 24],
                               labels=['Night', 'Morning', 'Midday', 'Evening', 'Late'])
ct = (df_multi.groupby(['hour_band', 'user_type', 'day_type'], observed=True)
              .size().reset_index(name='trips'))

g = sns.catplot(data=ct, x='hour_band', y='trips', hue='user_type', col='day_type',
                kind='bar', palette={'Customer': '#FF9800', 'Subscriber': '#2196F3'},
                height=5, aspect=1.2)
g.set_axis_labels('Hour Band', 'Number of Trips')
g.set_titles('{col_name}')
g.figure.suptitle('Trip Volume by Hour Band, User Type, and Day Type', y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

#### OBSERVATION: Weekday Subscribers show sharp dual peaks (8AM, 5PM commute).
#### Weekday Customers show a gentle midday bump. On weekends, BOTH user types
#### shift to a single midday peak (11AM-3PM), but Subscribers still outnumber
#### Customers. The commuter signal completely disappears on weekends.

### GOAL: Calculating average trip duration difference across hour and day for each user type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

for i, utype in enumerate(['Subscriber', 'Customer']):
    subset = df_multi[df_multi['user_type'] == utype]
    pivot = subset.pivot_table(index='day_of_week', columns='hour', 
                               values='duration_min', aggfunc='mean')
    pivot = pivot.reindex(day_order)
    
    sns.heatmap(pivot, ax=axes[i], cmap='YlOrRd', vmin=5, vmax=25,
                linewidths=0.5, cbar_kws={'label': 'Avg Duration (min)'})
    axes[i].set_title(f'{utype} — Avg Trip Duration', fontsize=13)
    axes[i].set_xlabel('Hour of Day')
    axes[i].set_ylabel('')

plt.tight_layout()
plt.show()

#### OBSERVATION: Subscribers have uniformly short trips (~8-10 min) across all
#### hours/days — true commuter consistency. Customers show higher duration 
#### especially during midday (10AM-3PM) and weekends, reaching 15-20+ minutes.
#### Late-night Customer trips are also longer (likely tourists/bar-goers).

### Talk about some of the relationships you observed in this part of the investigation. Were there features that strengthened each other in terms of looking at your feature(s) of interest?

> Yes — user type and day type strongly reinforce each other in explaining trip duration. Individually, user type shows Customers riding longer than Subscribers (median 12.65 vs 8.15 min). When combined with weekday/weekend, the effect amplifies: Customer median rises from 12.03 to 14.88 min (+23.7%) while Subscriber median holds flat at 8.15 to 8.08. The gap between groups widens from 3.88 to 6.80 min — a 75% increase at weekends. Volume moves the opposite way: normalised per day, Subscribers fall to 45.5% of their weekday rate while Customers hold at 94.1%. The Customer share of all trips nearly doubles from 9.0% to 17.1% at weekends. Duration and volume together describe habitual commuting versus discretionary leisure use — though the two groups remain overlapping populations (IQRs share ~4.2 min), not separable ones.

### Were there any interesting or surprising interactions between features?

> The most surprising finding was how little gender and age matter once user type is accounted for. I expected Female riders or older riders to have meaningfully different durations, but the box plots and scatter plots showed nearly identical patterns within the same user type. Gender adds only 1.05–1.64 min depending on user type — roughly 3–4x smaller than the user_type gap. Age explains just 0.07% of duration variance (Pearson r = 0.027).

> The 'Other' gender group (~2%) proved more nuanced than expected. It has the highest mean duration (11.42 min) but its median (9.07) sits below Female's (9.40). The mean is pulled up by a longer right tail, not by typical trips being longer. Stated on the median, the group falls between Female and Male rather than above both.

> The claim I had to revise was weekend convergence. I expected Subscribers to drift toward Customer-like durations at weekends, but the Subscriber median barely moves (8.15 → 8.08, a shift of 0.07 min). What does change is the Subscriber mean (9.72 → 10.49) — a fattening right tail, with the mean/median ratio rising from 1.19 to 1.30. A minority of Subscribers take longer weekend rides while the majority ride identically. On duration the groups actually diverge at weekends rather than converging; any convergence is in timing (midday peak), not trip length.

## Summary of Findings

**Key Findings:**

1. **Trip duration is right-skewed and short** — Median ride is 8.2 minutes, mean is 10.4. Roughly 92% of trips finish within 20 minutes. Mean exceeds median by a factor of ~1.2 in every subgroup, so the skew is a property of bike-share trips generally.

2. **User type is the dominant driver — Customers ride 55% longer** — Subscribers (89.6% of trips) have a median of 8.15 min; Customers (10.4%) have 12.65 min — a gap of 4.50 min. The distributions overlap substantially (IQRs share ~4.2 min), so this is a reliable shift, not a clean partition.

3. **Clear commuter-dominated system** — Dual peaks at 8AM and 5PM on weekdays; Thursday is busiest. Weekends shift to a single midday peak with far fewer Subscriber trips.

4. **Duration and volume move in opposite directions at weekends** — Customer median rises from 12.03 to 14.88 min (+23.7%); Subscriber median holds flat at 8.15 to 8.08. The gap between groups widens from 3.88 to 6.80 min (+75%). Meanwhile Subscriber volume drops to 45.5% of weekday rate while Customers hold at 94.1%. The groups diverge on duration at weekends, not converge.

5. **Demographics add almost nothing** — Age explains 0.07% of duration variance (r = 0.027). Gender produces a consistent but small effect: 1.05–1.64 min, roughly 3–4x smaller than the user-type gap. Neither reveals a hidden effect when user type is controlled.

6. **Rider base is young and male-dominated** — 75% Male, 23% Female, 2% Other. Peak age group is 25–35. Median rider age is 32.

**Data Cleaning Performed:**
- Removed trips > 60 minutes (1,710 trips, <1% — likely undocked/forgotten bikes)
- Removed ages > 80 (84 records — implausible birth years)
- Retained missing demographics (8,265 rows) — dropping them would bias against Customers, since 16.29% of Customer trips lack a birth year vs 3.07% of Subscriber trips
- Engineered features: `duration_min`, `member_age`, `hour`, `day_of_week`, `day_type`

**Main Takeaway:** Ford GoBike is used by two overlapping populations with reliably different behaviour — weekday commuters (Subscribers) and leisure riders (Customers). User type shifts the duration distribution by 4.50 min at the median and explains far more variation than age, gender, or any other feature. The two groups are not separable — roughly a quarter of Customer trips are shorter than the typical Subscriber trip — but their centres, timing, and weekend responses are consistently distinct.